In [0]:
import json
from pathlib import Path
import pandas as pd

In [0]:
dbutils.widgets.text("raw_root","/Volumes/workspace/raw/usgs_earthquakes", "Raw Root")

raw_root = Path(dbutils.widgets.get("raw_root"))

print("Ruta Raw: ", raw_root)

In [0]:
current_catalog = spark.sql("SELECT current_catalog()").first()[0]

catalog = current_catalog

schema = "bronze"

spark.sql(f""" CREATE SCHEMA IF NOT EXISTS {catalog}.{schema} COMMENT 'Datos crudos homogenizados en formato Delta' """)

print(f"Schema creado: {catalog}.{schema}")

In [0]:
cantidad_files = sorted(raw_root.glob("**/*.json"))

print("Archivos encontrados: ", len(cantidad_files))

for file in cantidad_files:
  print(file)

In [0]:
#dataframes = []

#for file in cantidad_files:
#    with open(file, "r", encoding="utf-8") as f:
#        payload = json.load(f)
#
#       df = pd.json_normalize(payload["features"])
#
#        df["_source_file"] = str(file)
#
#        if "/peru/" in str(file):
#            df["_extraccion_zona"] = "PERU"
#        elif "/chile/" in str(file):
#            df["_extraccion_zona"] = "CHILE"
#
#        dataframes.append(df)

#bronze_df = pd.concat(dataframes, ignore_index=True)

#print("cantidad de registros: ", len(bronze_df))
#print("cantidad de columnas: ", len(bronze_df.columns))


In [0]:
import pandas as pd
import json

dataframes = []

for file in cantidad_files:
    with open(file, "r", encoding="utf-8") as f:
        payload = json.load(f)

    df = pd.json_normalize(
        payload["features"]
    )
    df["_source_file"] = str(file)

    dataframes.append(df)

bronze_df = pd.concat(
    dataframes,
    ignore_index=True
)

print("Cantidad registros:", len(bronze_df))
print("Cantidad columnas:", len(bronze_df.columns))

In [0]:
spark_bronze_df = spark.createDataFrame(bronze_df)

print("registros: ", spark_bronze_df.count())
print("columnas: ", len(spark_bronze_df.columns))

#print(spark_bronze_df.head(5))
display(spark_bronze_df)

In [0]:
spark_bronze_df.printSchema()

In [0]:
from pyspark.sql.functions import col

bronze_final_df = spark_bronze_df.select(

    col("type").cast("string").alias("type"),
    col("id").cast("string").alias("id"),
    col("`properties.mag`").cast("double").alias("properties_mag"),
    col("`properties.place`").cast("string").alias("properties_place"),
    col("`properties.time`").cast("long").alias("properties_time"),
    col("`properties.updated`").cast("long").alias("properties_updated"),
    col("`properties.tz`").cast("string").alias("properties_tz"),
    col("`properties.url`").cast("string").alias("properties_url"),
    col("`properties.detail`").cast("string").alias("properties_detail"),
    col("`properties.felt`").cast("long").alias("properties_felt"),
    col("`properties.cdi`").cast("double").alias("properties_cdi"),
    col("`properties.mmi`").cast("double").alias("properties_mmi"),
    col("`properties.alert`").cast("string").alias("properties_alert"),
    col("`properties.status`").cast("string").alias("properties_status"),
    col("`properties.tsunami`").cast("long").alias("properties_tsunami"),
    col("`properties.sig`").cast("long").alias("properties_sig"),
    col("`properties.net`").cast("string").alias("properties_net"),
    col("`properties.code`").cast("string").alias("properties_code"),
    col("`properties.ids`").cast("string").alias("properties_ids"),
    col("`properties.sources`").cast("string").alias("properties_sources"),
    col("`properties.types`").cast("string").alias("properties_types"),
    col("`properties.nst`").cast("long").alias("properties_nst"),
    col("`properties.dmin`").cast("double").alias("properties_dmin"),
    col("`properties.rms`").cast("double").alias("properties_rms"),
    col("`properties.gap`").cast("double").alias("properties_gap"),
    col("`properties.magType`").cast("string").alias("properties_magType"),
    col("`properties.type`").cast("string").alias("properties_type"),
    col("`properties.title`").cast("string").alias("properties_title"),
    col("`geometry.type`").cast("string").alias("geometry_type"),
    col("`geometry.coordinates`").alias("geometry_coordinates"),
    col("_source_file").cast("string").alias("_source_file")
)

bronze_final_df.printSchema()


In [0]:
#%sql
#DROP TABLE IF EXISTS workspace.bronze.earthquakes;

In [0]:
table_name = f"{catalog}.{schema}.earthquakes"

spark.sql(f"""
CREATE TABLE IF NOT EXISTS {table_name} (

    type STRING,
    id STRING,
    properties_mag DOUBLE,
    properties_place STRING,
    properties_time BIGINT,
    properties_updated BIGINT,
    properties_tz STRING,
    properties_url STRING,
    properties_detail STRING,
    properties_felt BIGINT,
    properties_cdi DOUBLE,
    properties_mmi DOUBLE,
    properties_alert STRING,
    properties_status STRING,
    properties_tsunami BIGINT,
    properties_sig BIGINT,
    properties_net STRING,
    properties_code STRING,
    properties_ids STRING,
    properties_sources STRING,
    properties_types STRING,
    properties_nst BIGINT,
    properties_dmin DOUBLE,
    properties_rms DOUBLE,
    properties_gap DOUBLE,
    properties_magType STRING,
    properties_type STRING,
    properties_title STRING,
    geometry_type STRING,
    geometry_coordinates ARRAY<DOUBLE>,
    _source_file STRING

)
USING DELTA
""")

print(f"Tabla Delta creada: {table_name}")

In [0]:
(
    bronze_final_df.write.format("delta").mode("overwrite").saveAsTable("workspace.bronze.earthquakes")
)

print(f"Datos guardados en: {table_name}")